# Test when a higher-timeframe feature becomes visible

The fixed M1 and M5 fixtures expose the boundary at the M5 close. Add a later M5 bar to check that it leaves the earlier rows unchanged.

In [1]:
import pandas as pd

from market_data.transforms.align import align_timeframes

minute_index = pd.date_range("2025-01-06 14:34", periods=3, freq="min", tz="UTC")
m1 = pd.DataFrame({"close": [99.0, 101.0, 102.0]}, index=minute_index)
m5_index = pd.date_range("2025-01-06 14:30", periods=2, freq="5min", tz="UTC")
m5 = pd.DataFrame({"close": [100.0, 200.0]}, index=m5_index)
availability = pd.Series(m5_index + pd.Timedelta(minutes=5), index=m5_index)

aligned = align_timeframes(
    m1, {"M5": m5.iloc[:1]}, availability_times={"M5": availability.iloc[:1]}
)
values = aligned["M5_close"]
assert pd.isna(values.iloc[0])
assert values.iloc[1:].tolist() == [100.0, 100.0]

longer_input = align_timeframes(m1, {"M5": m5}, availability_times={"M5": availability})
pd.testing.assert_frame_equal(aligned, longer_input)
values.to_frame()

,M5_close
2025-01-06 14:34:00+00:00,NaN
2025-01-06 14:35:00+00:00,100.0
2025-01-06 14:36:00+00:00,100.0
